# Meetlat voor embeddings

Doel: meten of een embeddingmodel de taal van de rapportages beter "verstaat". Dit notebook draait lokaal en meet elk model op dezelfde manier, zodat we stap voor stap kunnen vergelijken.

## Het plan

We oefenen op de Fruta-data. Straks doen we hetzelfde in Snowflake op de echte rapportages (12 miljoen).

| Stap | Wat | Waar |
|---|---|---|
| 0 | **Meetlat + nulmeting** (dit notebook) met het huidige model `snowflake-arctic-embed-l-v2.0` | lokaal |
| 1 | **TSDAE**: het model unsupervised aanpassen aan onze taal. Woorden uit een rapportage weghalen en het model de zin laten reconstrueren vanuit één embedding. Getraind op de teksten uit `RECORDS`, zonder labels. `NOTES` blijft buiten de training, die gebruiken we alleen om te meten. Het model komt in `models/`, buiten git. | lokaal, nieuw notebook |
| 2 | **Opnieuw meten** met dit notebook (`MODEL_NAME` = pad naar het nieuwe model) en vergelijken met de nulmeting | lokaal |
| 3 | Alleen als het beter is: embeddings opnieuw maken (`02`), UMAP opnieuw (`03`), nieuwe achtergrondkaart | lokaal + Snowflake |
| 4 | **BERTopic** op de nieuwe embeddings: onderwerpen vinden zonder labels. De NMI hieronder meet al hoe goed clusters met de categorieën samenvallen; bij BERTopic gebruiken we diezelfde maat. | lokaal |
| 5 | **Opschalen naar Snowflake** met de echte data, zie hieronder | Snowflake |

### Bij stap 5: wat anders wordt bij de echte data
- **Geen categorieën.** De echte rapportages hebben geen `CATEGORY`. Voor de meetlat is een kleine set met de hand gelabelde rapportages nodig, een paar honderd, met dezelfde soort onderwerpen als in `NOTES`.
- **Trainen op een steekproef.** Voor TSDAE zijn een paar honderdduizend rapportages ruim genoeg, niet alle 12 miljoen.
- **GPU nodig.** Het zware werk zijn de embeddings van 12 miljoen rapportages. Daarvoor is een container-notebook op een GPU-compute-pool nodig, en dat kan niet op een trialaccount.
- **Eigen model voor vragen.** Een eigen model werkt niet met `AI_EMBED`. Ook vragen moeten met het eigen model worden omgezet, bijvoorbeeld via de Model Registry als service.

## De maten

Allemaal gemeten op `NOTES`, de voorbeeldrapportages met een categorie. Hoger is beter.

| Maat | Wat het meet |
|---|---|
| `KNN_PURITY` | Van de 5 meest gelijkende notes: welk deel heeft dezelfde categorie? |
| `KNN_ACCURACY` | Hoe vaak voorspellen de 5 dichtstbijzijnde notes de juiste categorie (5-voudige kruisvalidatie)? |
| `SILHOUETTE` | Hoe goed de categorieën als losse groepen in de ruimte liggen (-1 tot 1) |
| `NMI` | Hoe goed KMeans-clusters met de categorieën samenvallen (0 tot 1). Relevant voor BERTopic. |
| `QUERY_PRECISION` | Bij voorbeeldvragen: welk deel van de top 10 heeft de verwachte categorie? |

Elke meting wordt bewaard in `FRUTA.PUBLIC.EMBEDDING_EVAL`, zodat de runs naast elkaar te zien zijn.

In [ ]:
import datetime as dt
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from snowflake.snowpark import Session

# Welk model meten we? Nulmeting: het basismodel. Later: het pad naar het getrainde model, bijv. "../models/arctic-tsdae"
MODEL_NAME = "Snowflake/snowflake-arctic-embed-l-v2.0"
RUN_NAME = "nulmeting"

K = 5  # aantal buren voor KNN_PURITY en KNN_ACCURACY
TOP_N = 10  # top-n voor QUERY_PRECISION
RANDOM_STATE = 0

# Voorbeeldvragen met de categorie die we in de top verwachten. Vul gerust aan.
QUERIES = {
    "Hulp bij wassen en aankleden": "adl",
    "Mevrouw sliep slecht en was 's nachts onrustig": "nachten",
    "Wond aan de stuit, verband verwisseld": "huid",
    "Meneer loopt moeizaam met de rollator": "mobiliteit",
    "Gezellige activiteit in de huiskamer": "sociaal",
}

load_dotenv()
session = Session.builder.configs(
    {
        "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        "user": os.getenv("SNOWFLAKE_USER"),
        "password": os.getenv("SNOWFLAKE_PASSWORD"),
        "warehouse": "COMPUTE_WH",
        "database": "FRUTA",
        "schema": "PUBLIC",
    }
).create()

In [ ]:
# Haal de NOTES op en maak de embeddings met het gekozen model
notes_df = session.sql(
    "SELECT ID, CATEGORY, NOTE FROM NOTES WHERE NOTE IS NOT NULL AND NOTE != '' ORDER BY ID"
).to_pandas()
labels = notes_df["CATEGORY"].to_numpy()
print(f"{len(notes_df)} notes in {notes_df['CATEGORY'].nunique()} categorieën.")

model = SentenceTransformer(MODEL_NAME)
embs = model.encode(
    notes_df["NOTE"].tolist(), batch_size=32, show_progress_bar=True, normalize_embeddings=True
)

In [ ]:
# KNN_PURITY: de note zelf telt niet mee als buur
_, idx = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(embs).kneighbors(embs)
same_category = labels[idx[:, 1:]] == labels[:, None]
knn_purity = same_category.mean()

# KNN_ACCURACY: categorie voorspellen uit de K dichtstbijzijnde buren
knn_accuracy = cross_val_score(
    KNeighborsClassifier(n_neighbors=K, metric="cosine"),
    embs,
    labels,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
).mean()

# SILHOUETTE: op een steekproef, want de berekening groeit kwadratisch met het aantal notes
silhouette = silhouette_score(
    embs, labels, metric="cosine", sample_size=min(len(embs), 5000), random_state=RANDOM_STATE
)

# NMI: KMeans met evenveel clusters als categorieën
clusters = KMeans(
    n_clusters=notes_df["CATEGORY"].nunique(), n_init=10, random_state=RANDOM_STATE
).fit_predict(embs)
nmi = normalized_mutual_info_score(labels, clusters)

In [ ]:
# QUERY_PRECISION. Arctic verwacht bij zoekvragen de prompt 'query'; een eigen model heeft die misschien niet.
prompt_name = "query" if "query" in model.prompts else None
query_embs = model.encode(list(QUERIES), prompt_name=prompt_name, normalize_embeddings=True)
top = np.argsort(-(query_embs @ embs.T), axis=1)[:, :TOP_N]

query_results = pd.DataFrame(
    {
        "vraag": list(QUERIES),
        "verwacht": list(QUERIES.values()),
        "precisie": (labels[top] == np.array(list(QUERIES.values()))[:, None]).mean(axis=1),
    }
)
query_precision = query_results["precisie"].mean()
query_results

In [ ]:
# Per categorie: waar is het model goed en waar minder?
pd.DataFrame({"CATEGORY": labels, "KNN_PURITY": same_category.mean(axis=1)}).groupby(
    "CATEGORY"
).agg(AANTAL=("KNN_PURITY", "size"), KNN_PURITY=("KNN_PURITY", "mean")).sort_values(
    "KNN_PURITY"
)

In [ ]:
# Kijk zelf: de top van de eerste vraag
first_query = list(QUERIES)[0]
notes_df.iloc[top[0]][["CATEGORY", "NOTE"]].assign(vraag=first_query)

In [ ]:
# Bewaar de meting in Snowflake, zodat runs te vergelijken zijn
result = pd.DataFrame(
    [
        {
            "RUN_NAME": RUN_NAME,
            "MODEL": MODEL_NAME,
            "N_NOTES": len(notes_df),
            "KNN_PURITY": knn_purity,
            "KNN_ACCURACY": knn_accuracy,
            "SILHOUETTE": silhouette,
            "NMI": nmi,
            "QUERY_PRECISION": query_precision,
            "EVALUATED_AT": dt.datetime.now().isoformat(timespec="seconds"),
        }
    ]
)
session.write_pandas(
    result, "EMBEDDING_EVAL", auto_create_table=True, overwrite=False, quote_identifiers=False
)
result

In [ ]:
# Alle metingen tot nu toe
session.sql("SELECT * FROM EMBEDDING_EVAL ORDER BY EVALUATED_AT").to_pandas()

In [ ]:
session.close()